In [ ]:
    import pandas as pd
import numpy as np
import matplotlib.pyplot as plt



plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

def load_dataset(filename):
    BASE_DIR = '../data'
    EXTENSION = '.csv'
    TARGET_PATH = f'{BASE_DIR}/{filename}{EXTENSION}'
    try:
        return pd.read_csv(TARGET_PATH)
    except FileNotFoundError:
        return None
    except UnicodeDecodeError:
        return pd.read_csv(TARGET_PATH, encoding = 'euc-kr')

In [ ]:
port1 = load_dataset('port1')
record = load_dataset('record')

In [ ]:
port = port1.copy()
rec = record.copy()

port["접안시각"] = pd.to_datetime(
    port["접안(예정)일시"], format="mixed", errors="coerce"
)

port["선석종료시각"] = pd.to_datetime(
    port["출항(예정)일시"], format="mixed", errors="coerce"
)

rec["입항시각"] = pd.to_datetime(
    rec["입항일시"], format="mixed", errors="coerce"
)

rec["출항시각"] = pd.to_datetime(
    rec["출항일시"], format="mixed", errors="coerce"
)

In [ ]:
# 선석에 붙어 있던 시간
port["선석사용시간"] = (
    port["선석종료시각"] - port["접안시각"]
).dt.total_seconds() / 3600

# 항구에 들어와서 나갈 때까지의 시간
rec["계산재항시간"] = (
    rec["출항시각"] - rec["입항시각"]
).dt.total_seconds() / 3600

display(
    port[
        ["선명", "선석", "접안시각", "선석종료시각", "선석사용시간"]
    ].head()
)

display(
    rec[
        ["선명", "계선장소", "입항시각", "출항시각", "계산재항시간"]
    ].head()
)

In [ ]:
# record 자료에서 신항 1부두만 선택
rec = rec[
    rec["계선장소"].str.match(
        r"^신항 1부두 [123]선석$", na=False
    )
].copy()

# 이름의 공백과 대소문자 차이를 없앰
port["연결선명"] = (
    port["선명"].str.upper().str.replace(r"\s+", "", regex=True)
)

rec["연결선명"] = (
    rec["선명"].str.upper().str.replace(r"\s+", "", regex=True)
)

# T1(P) → "1", 신항 1부두 1선석 → "1"
port["선석번호"] = port["선석"].str.extract(r"T([123])")[0]
rec["선석번호"] = rec["계선장소"].str.extract(r"([123])선석")[0]

# 같은 기록이 두 번 연결되는지 확인할 번호
rec["기록번호"] = range(len(rec))

In [ ]:
port = port.dropna(
    subset=["선석번호", "접안시각", "선석종료시각"]
).copy()

rec = rec.dropna(
    subset=["선석번호", "입항시각", "출항시각"]
).copy()

In [ ]:
ships = pd.merge_asof(
    # 두 표를 각각 출항 시각순으로 정렬
    port.sort_values("선석종료시각"),

    rec[
        ["연결선명", "선석번호", "입항시각", "출항시각", "기록번호"]
    ].sort_values("출항시각"),

    left_on="선석종료시각",
    right_on="출항시각",

    # 선박 이름과 선석 번호가 같아야 연결
    by=["연결선명", "선석번호"],

    # 가장 가까운 출항 시각을 찾음
    direction="nearest",

    # 두 출항 시각의 차이가 24시간 이내인 경우만 연결
    tolerance=pd.Timedelta(hours=24)
)

In [ ]:
# 연결하지 못한 행 제외
ships = ships.dropna(subset=["기록번호"]).copy()

# 같은 record가 여러 번 연결되면 시각 차이가 가장 작은 것만 사용
ships["출항시각차이"] = (
    ships["선석종료시각"] - ships["출항시각"]
).abs()

ships = (
    ships.sort_values("출항시각차이")
    .drop_duplicates("기록번호")
)

# 입항 후 접안까지 걸린 시간
ships["접안전시간"] = (
    ships["접안시각"] - ships["입항시각"]
).dt.total_seconds() / 3600

# 항구에 머문 전체 시간
ships["재항시간"] = (
    ships["출항시각"] - ships["입항시각"]
).dt.total_seconds() / 3600

# 음수 시간 등 계산에 사용할 수 없는 기록 제외
ships = ships[
    (ships["접안전시간"] >= 0)
    & (ships["선석사용시간"] > 0)
    & (ships["재항시간"] > 0)
].copy()

ships = ships.sort_values("입항시각").reset_index(drop=True)

In [ ]:
print("계산에 사용할 입항 건수:", len(ships))

display(
    ships[
        [
            "선명",
            "선석번호",
            "입항시각",
            "접안시각",
            "선석사용시간",
            "접안전시간",
            "재항시간"
        ]
    ].head()
)

In [ ]:
# 2022~2025년에 입항한 배만 선택
sim_data = ships[
    (ships["입항시각"] >= "2022-01-01")
    & (ships["입항시각"] < "2026-01-01")
].copy()

sim_data = sim_data.sort_values("입항시각")

# 분석 시작과 끝
period_start = pd.Timestamp("2022-01-01")
period_end = pd.Timestamp("2026-01-01")

# 4년간 시간 수 — 2024년 윤년도 자동 반영
period_hours = (
    period_end - period_start
).total_seconds() / 3600

# 입항 시각을 '2022년 시작부터 몇 시간 지났는지'로 변환
sim_data["입항경과시간"] = (
    sim_data["입항시각"] - period_start
).dt.total_seconds() / 3600

# 선석을 떠난 뒤 항구에서 나가기까지의 추가 시간
sim_data["출항추가시간"] = (
    sim_data["출항시각"] - sim_data["선석종료시각"]
).dt.total_seconds().div(3600).clip(lower=0)

# 계산에 필요한 네 항목을 목록으로 준비
배정자료 = list(
    sim_data[
        ["입항경과시간", "선석번호", "선석사용시간", "출항추가시간"]
    ].itertuples(index=False, name=None)
)

In [ ]:
def simulate(scale):
    # 선석별로 다시 사용할 수 있는 시각
    free = {1: 0.0, 2: 0.0, 3: 0.0}

    waits = []       # 배마다 기다린 시간
    residences = [] # 배마다 항구에 머문 시간
    occupied = 0.0   # 선석을 사용한 총시간

    for arrival, berth, service, exit_time in 배정자료:
        berth = int(berth)

        # 선석 사용시간에 배율 적용
        work = service * scale

        # 배 도착과 선석이 비는 시각 중 늦은 때에 접안
        start = max(arrival, free[berth])

        # 작업이 끝나면 선석이 다시 빔
        finish = start + work
        free[berth] = finish

        # 대기시간과 재항시간 기록
        waits.append(start - arrival)
        residences.append(finish + exit_time - arrival)

        # 2022~2025년 안에서 사용한 시간만 집계
        occupied += max(
            0,
            min(finish, period_hours) - max(start, 0)
        )

    return {
        "사용시간배율": scale,
        "점유율": occupied / (3 * period_hours) * 100,
        "평균대기시간": sum(waits) / len(waits),
        "평균재항시간": sum(residences) / len(residences)
    }

In [ ]:
baseline = simulate(1.0)

display(pd.DataFrame([baseline]).round(2))

In [ ]:
# 0.80배부터 1.15배까지 141번 계산
scales = [0.80 + i * 0.0025 for i in range(141)]

results = pd.DataFrame(
    [simulate(scale) for scale in scales]
)

display(results.head())

In [ ]:
import matplotlib.pyplot as plt

# 윈도우 한글 글꼴
plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# 왼쪽: 점유율과 평균 대기시간
axes[0].plot(
    results["점유율"],
    results["평균대기시간"],
    color="royalblue",
    linewidth=2
)

axes[0].axhline(
    3,
    color="tomato",
    linestyle="--",
    label="평균 대기 3시간"
)

axes[0].set_title("점유율과 평균 대기시간")
axes[0].set_xlabel("선석 점유율 (%)")
axes[0].set_ylabel("평균 대기시간 (시간)")
axes[0].set_ylim(bottom=0)
axes[0].legend()
axes[0].grid(alpha=0.3)

# 오른쪽: 점유율과 평균 재항시간
axes[1].plot(
    results["점유율"],
    results["평균재항시간"],
    color="seagreen",
    linewidth=2
)

axes[1].set_title("점유율과 평균 재항시간")
axes[1].set_xlabel("선석 점유율 (%)")
axes[1].set_ylabel("평균 재항시간 (시간)")
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# 대기시간이 3시간보다 작은 마지막 점
below = results[
    results["평균대기시간"] < 3
].iloc[-1]

# 대기시간이 3시간 이상인 첫 번째 점
above = results[
    results["평균대기시간"] >= 3
].iloc[0]

# 두 점 사이에서 3시간이 위치하는 비율
ratio = (
    (3 - below["평균대기시간"])
    / (above["평균대기시간"] - below["평균대기시간"])
)

# 그 위치에 해당하는 점유율
occupancy_at_3h = (
    below["점유율"]
    + ratio * (above["점유율"] - below["점유율"])
)

print(f"평균 대기 3시간에 해당하는 모형 점유율: {occupancy_at_3h:.1f}%")